# Fine-tune `cuplm` on a free GPU (Colab)

One-time job, ~15-30 min, **$0** on Colab's free T4.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then set `REPO_URL` below to your pushed GitHub repo (e.g. `https://github.com/you/cuplm`).

In [ ]:
REPO_URL = "https://github.com/REPLACE_ME/cuplm"  # <- set this
!nvidia-smi -L  # confirm a GPU is attached

In [ ]:
!git clone -q $REPO_URL cuplm
%cd cuplm
!pip install -q -e .
!pip install -q -r training/requirements.txt
!pip install -q gguf

In [ ]:
# 1. Build the self-verifying dataset, 2. sanity-check the label masking
!python -m training.generate_data
!python -m training.train_qlora --validate

In [ ]:
# 3. QLoRA fine-tune (4-bit on the GPU). ~15-30 min on a T4.
!python -m training.train_qlora \
    --model Qwen/Qwen3-0.6B \
    --data training/data/cuplm_sft.jsonl \
    --out cuplm-lora \
    --epochs 3 --batch 8 --4bit

In [ ]:
# 4. Merge the adapter into full weights
!python -m training.export_gguf --base Qwen/Qwen3-0.6B --adapter cuplm-lora --out merged/cuplm

In [ ]:
# 5. Convert to GGUF (Q8_0 - good quality for a 0.6B, no extra quantizer needed)
!git clone -q https://github.com/ggml-org/llama.cpp
!python llama.cpp/convert_hf_to_gguf.py merged/cuplm \
    --outfile cuplm-v0-Q8_0.gguf --outtype q8_0
import os; print('size MB:', round(os.path.getsize('cuplm-v0-Q8_0.gguf')/1e6, 1))

In [ ]:
# 6. Download it to your PC, drop it in cuplm/models/, then:
#    llama-server -m models/cuplm-v0-Q8_0.gguf -c 4096 -t 2 --port 8080
#    cup chat --no-think --grammar
#    python -m bench.run_bench --server http://localhost:8080   # beat 4/6!
from google.colab import files
files.download('cuplm-v0-Q8_0.gguf')